# 4. First Escalation: Retrieval (RAG)
## From Documents to Usable Context — Adding context without changing the model

**Purpose:** Help participants internalize that getting documents into text is not enough. Value only appears once documents become retrievable, bounded, and evaluable context that a model can reason over.

This is where we move from document plumbing to AI system behavior.

---

### Prerequisites
- Section 2 complete (API key and endpoint configured)
- Section 3 complete (`Basic-Fantasy-RPG-Rules-r142.md` generated by Docling)
- Embedding model pre-downloaded to `./models/granite-embedding-30m-english`


## 4.1 The Real Problem: "We Have Text. Now What?"

After extraction, teams often experience a brief sense of victory.

> "The document is in Markdown. We're done."

But text alone does not create intelligence. A common first instinct is the **"just add it to context"** approach: take the entire document, paste it into the prompt, ask the model a question.

We already proved in the previous notebook that this fails — the document exceeds the deployed context window (120k tokens). Even when we force it to fit by truncating, we're paying a massive token cost for a simple question.

**The model doesn't need the whole book. It needs the right paragraph.**

That's retrieval.


## 4.2 Setup


In [7]:
! pip install chromadb sentence-transformers tiktoken -q

In [8]:
# Environment fix: disable tqdm progress bars to avoid Jupyter thread lock issues
import os
os.environ["TQDM_DISABLE"] = "1"

import requests
import json

In [9]:
# Carry over your credentials from Section 2
# Replace these if you're running this notebook standalone
key = "sk-UFHcLOTZk_73o6YwVQhSiQ"
endpoint_base = "https://litellm-prod.apps.maas.redhatworkshops.io/v1"

url_chat = f"{endpoint_base}/chat/completions"

# Quick connection test
headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
test_data = {"model": "granite-3-2-8b-instruct", "messages": [{"role": "user", "content": "ping"}], "max_tokens": 5}
resp = requests.post(url_chat, headers=headers, json=test_data)
if resp.status_code == 200:
    print("✅ API connection working")
else:
    print(f"❌ API connection failed: {resp.status_code}")

✅ API connection working


## 4.3 Chunking Is a Design Decision, Not a Default

If retrieval is about selecting the right context, **chunking defines what "right" can even mean.**

When we split documents into chunks, we are making architectural choices:
- How much context travels together
- What semantic boundaries are preserved
- What the retriever can return
- What the model can reason over in a single pass

**A chunk is not just a slice of text. It is a unit of reasoning.**

> **Field takeaway:** Chunking strategy is a customer conversation, not a backend detail.


### 4.3.1 Load the Docling Output

We pick up where Section 3 left off. The markdown file is the **contract** between ingestion and retrieval.


In [10]:
with open("Basic-Fantasy-RPG-Rules-r142.md", "r", encoding="utf-8") as f:
    full_text = f.read()

print(f"Document length: {len(full_text):,} characters")
print(f"\nFirst 500 characters:\n{full_text[:500]}...")

Document length: 908,803 characters

First 500 characters:
<!-- image -->

Copyright © 2006-2025 Chris Gonnerman All Rights Reserved.  See next page for license information. www.basicfantasy.org

Dedicated to Gary Gygax, Dave Arneson, Tom Moldvay, David Cook, and Stephen Marsh and to my daughter Taylor, my first and best inspiration

## Basic Fantasy Role-Playing Game

4th Edition, Release 142

Copyright © 2006-2025 Chris Gonnerman - All Rights Reserved

<!-- image -->

All textual materials in this document, as well as all maps, floorplans, diagrams, c...


### 4.3.2 Define the Chunking Function

We use a simple **fixed-size chunker with overlap.** This is the most common starting point.

**Key parameters:**
- `chunk_size` — How many characters per chunk. Larger = more context per chunk, but less precision. Smaller = more precise retrieval, but risks cutting mid-thought.
- `overlap` — Characters shared between adjacent chunks. Prevents information from being lost at chunk boundaries. If a rule spans two chunks, overlap ensures at least one chunk has the complete rule.


In [13]:
def chunk_text(text, chunk_size=500, overlap=100):
    """
    Split text into overlapping chunks.
    
    Args:
        text: The full document text
        chunk_size: Target size of each chunk in characters
        overlap: Number of characters shared between adjacent chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


chunks = chunk_text(full_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")

Total chunks created: 1137
Average chunk length: 999 characters


### 4.3.3 Inspect the Chunks

Before we embed anything, **look at what we produced.** Where did the splits land? Did we cut in the middle of a rule? A table? A sentence?

This is what we mean by "chunking is a design decision."


In [14]:
print("--- Chunk 0 ---")
print(chunks[0])
print(f"\n--- Chunk 1 ---")
print(chunks[1])

print(f"\n{'=' * 60}")
print("OVERLAP REGION")
print(f"{'=' * 60}")
print(f"End of chunk 0:   ...{chunks[0][-120:]}")
print(f"Start of chunk 1: {chunks[1][:120]}...")

--- Chunk 0 ---
<!-- image -->

Copyright © 2006-2025 Chris Gonnerman All Rights Reserved.  See next page for license information. www.basicfantasy.org

Dedicated to Gary Gygax, Dave Arneson, Tom Moldvay, David Cook, and Stephen Marsh and to my daughter Taylor, my first and best inspiration

## Basic Fantasy Role-Playing Game

4th Edition, Release 142

Copyright © 2006-2025 Chris Gonnerman - All Rights Reserved

<!-- image -->

All textual materials in this document, as well as all maps, floorplans, diagrams, charts, and forms included herein, are distributed under the terms of the Creative Commons Attribution-ShareAlike 4.0 International

<!-- image -->

License .   Most other artwork presented is property of the original artist and is used with permission. Note that you may not publish or otherwise distribute this work as is without permission of the original artists; you must remove all non-licensed artwork before doing so.

The full license text can be viewed at:

https://creativec

In [15]:
# Find the chunk(s) containing our test case: the Thief's Open Locks rule
for i, chunk in enumerate(chunks):
    if "open locks" in chunk.lower():
        print(f"\n=== Chunk {i} (contains 'Open Locks') ===")
        print(chunk)
        print(f"Length: {len(chunk)} chars")
        print()


=== Chunk 71 (contains 'Open Locks') ===
t   the   GM   may   apply situational   adjustments   (plus   or   minus   percentage points) as they see fit; for instance, it's obviously harder to climb a wall slick with slime than one that is dry, so the GM might apply a penalty of 20% for the slimy wall.

## BASIC FANTASY RPG

## Thief Abilities

|   Thief Level |   Open Locks |   Remove Traps |   Pick Pockets |   Move Silently |   Climb Walls |   Hide |   Listen |
|---------------|--------------|----------------|----------------|-----------------|---------------|--------|----------|
|             1 |           25 |             20 |             30 |              25 |            80 |     10 |       30 |
|             2 |           30 |             25 |             35 |              30 |            81 |     15 |       34 |
|             3 |           35 |             30 |             40 |              35 |            82 |     20 |       38 |
|             4 |           40 |             35 

### 4.3.4 Experiment: What Happens With Different Chunk Sizes?

This is **not optimization.** This is understanding the tradeoff.

Smaller chunks → more of them → more precise retrieval → but each chunk has less context.  
Larger chunks → fewer of them → each retrieval returns more text → but retrieval is less targeted.


In [16]:
for size in [200, 500, 1000, 2000]:
    test_chunks = chunk_text(full_text, chunk_size=size, overlap=int(size * 0.2))
    print(f"Chunk size: {size:>5}  |  Overlap: {int(size*0.2):>4}  |  Total chunks: {len(test_chunks):>4}")

Chunk size:   200  |  Overlap:   40  |  Total chunks: 5681
Chunk size:   500  |  Overlap:  100  |  Total chunks: 2273
Chunk size:  1000  |  Overlap:  200  |  Total chunks: 1137
Chunk size:  2000  |  Overlap:  400  |  Total chunks:  569


## 4.4 Embeddings and the Vector Store

Now we turn our chunks into something **searchable.**

### Two Models, Two Jobs

A RAG system uses **two separate models:**
1. **Embedding model** — converts text into numerical vectors that capture semantic meaning. Chunks about similar topics get similar vectors, even if they use different words.
2. **Language model** — generates the answer, using retrieved chunks as context. This is the same `granite-3-2-8b-instruct` we've been using.

The embedding model is small and fast. It doesn't generate text — it just maps text to numbers.

### What's a Vector Store?

A vector store holds all the chunk embeddings and provides **fast similarity search.** When a question comes in:
1. Embed the question using the same embedding model
2. Find the chunks whose embeddings are closest to the question embedding
3. Return those chunks as context

> **We're using ChromaDB** — it runs inside this notebook, no servers needed.


### 4.4.1 Load the Embedding Model


In [17]:
from sentence_transformers import SentenceTransformer

print("Loading Granite embedding model...")
embed_model = SentenceTransformer("./models/granite-embedding-30m-english")
print("✅ Embedding model loaded")

Loading Granite embedding model...


/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


✅ Embedding model loaded


### 4.4.2 Embed All Chunks


In [18]:
print(f"Embedding {len(chunks)} chunks...")
embeddings = embed_model.encode(chunks, show_progress_bar=False, batch_size=64)
print(f"✅ Embeddings complete — shape: {embeddings.shape}")

Embedding 1137 chunks...
✅ Embeddings complete — shape: (1137, 384)


### 4.4.3 Load Into ChromaDB


In [20]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="rpg_rules",
    metadata={"hnsw:space": "cosine"}
)

print(f"Loading {len(chunks)} chunks into vector store...")
collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{
        "chunk_index": i,
        "source": "Basic-Fantasy-RPG-Rules-r142.md"
    } for i in range(len(chunks))]
)

print(f"✅ Loaded {collection.count()} chunks into the vector store")

InternalError: Collection [rpg_rules] already exists

> **Note:** We embedded the chunks ourselves and passed the pre-computed vectors to ChromaDB. This gives us full control over the embedding step and avoids compatibility issues with ChromaDB's built-in embedding wrappers.

> **Alternative:** If your MaaS instance has `nomic-embed-text-v1-5` available, you can use the API for embeddings instead of the local model. See the appendix at the end of this notebook.


## 4.5 Retrieval: Can We Find the Right Paragraph?

Before we involve the language model at all, let's test whether retrieval works.

> **Facilitator note:** This is a critical checkpoint. If retrieval doesn't return the right chunks, no model — no matter how powerful — can give a correct answer. Retrieval failure is the most common cause of RAG failure.


In [25]:
question = "What happens if a Thief fails an Open Locks attempt?"

# Embed the question with the same model used for chunks
question_embedding = embed_model.encode([question]).tolist()

# Retrieve the top 3 most relevant chunks
results = collection.query(
    query_embeddings=question_embedding,
    n_results=3
)

print(f"Question: {question}\n")
for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
    similarity = 1 - distance
    print(f"--- Retrieved Chunk {i+1} (similarity: {similarity:.3f}) ---")
    print(doc)
    print()

Question: What happens if a Thief fails an Open Locks attempt?

--- Retrieved Chunk 1 (similarity: 0.769) ---
ttempt fails, the Thief must wait until they have gained another level of experience before trying again.

Remove Traps is generally rolled twice: first to detect the trap, and second to disarm it.   The GM will make these rolls as the player won't know for sure if the character is successful or not until someone actually tests the trapped (or suspected) area.

Pick Pockets allows the Thief to lift the wallet, cut the purse, etc. of a victim without being noticed.   If the roll fails, the Thief didn't get what they wanted; but further, the intended victim (or an onlooker, at the GM's option) will notice the attempt if the die roll is more than two times the target number (or if the die roll is 00).

Move Silently , like Remove Traps, is always rolled by the GM.    The   Thief   will   usually   believe   they   are   moving silently regardless of the die roll, but opponents the

In [26]:
# Sanity check: is the actual answer in what we retrieved?
expected_phrase = "must wait until they have gained another level"
found_in_context = any(expected_phrase.lower() in doc.lower() for doc in results['documents'][0])

if found_in_context:
    print("✅ Retrieval found the relevant chunk. The model has what it needs.")
else:
    print("❌ Retrieval missed the relevant chunk. The model will likely fail.")
    print("   This is a retrieval problem, not a model problem.")

✅ Retrieval found the relevant chunk. The model has what it needs.


### What just happened?

We embedded the question with the **same Granite embedding model** we used for the chunks, then asked ChromaDB to find the closest matches using cosine similarity.

**We didn't search for keywords.** We searched for *meaning.* That's why a question about "fails an Open Locks attempt" can match a chunk that says "if the attempt fails" — even though the exact phrasing differs.

> **Key insight:** If the right chunk isn't in the retrieval results, everything downstream fails. Always inspect retrieval before blaming the model.


## 4.6 The Complete RAG Pipeline: Retrieve, Then Generate

Now we connect it all. The function below does three things:
1. **Retrieve** — find the most relevant chunks for the question
2. **Construct** — build a prompt with system instructions + retrieved context + question
3. **Generate** — send it to the language model for a grounded answer

> Notice: we tell the model explicitly to answer **only** from the provided context. This is how we get *grounded* answers — the quality the customer needs (Section 1.4).


In [27]:
def rag_query(question, collection, embed_model, api_key, base_url,
              model="granite-3-2-8b-instruct", n_results=3, max_tokens=300):
    """
    Full RAG pipeline: retrieve context, then generate a grounded answer.
    
    Args:
        question: The user's question
        collection: ChromaDB collection with pre-embedded chunks
        embed_model: SentenceTransformer model for embedding the question
        api_key: MaaS API key
        base_url: MaaS API base URL
        model: Model to use for generation
        n_results: Number of chunks to retrieve
        max_tokens: Max tokens in the generated answer
    
    Returns:
        dict with 'answer', 'retrieved_chunks', and 'model' keys
    """
    # Step 1: Embed the question and retrieve relevant chunks
    q_embedding = embed_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=q_embedding,
        n_results=n_results
    )
    retrieved_chunks = results['documents'][0]
    
    # Step 2: Build the grounded prompt
    context = "\n\n---\n\n".join(retrieved_chunks)
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the question using ONLY "
                "the provided context. If the context does not contain enough "
                "information to answer, say so explicitly. Do not make up information."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    
    # Step 3: Generate the answer
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0
    }
    
    url = f"{base_url}/chat/completions"
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    
    answer = response.json()["choices"][0]["message"]["content"]
    
    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "model": model
    }

### 4.6.1 Test It — Same Model, Same Question, Different Result

The model hasn't changed. The question hasn't changed. **The only difference is context.**


In [28]:
result = rag_query(
    question="What happens if a Thief fails an Open Locks attempt?",
    collection=collection,
    embed_model=embed_model,
    api_key=key,
    base_url=endpoint_base
)

print(f"Question: {result['question']}\n")
print(f"RAG Answer:\n{result['answer']}\n")
print(f"Model used: {result['model']}")
print(f"Context chunks used: {len(result['retrieved_chunks'])}")

Question: What happens if a Thief fails an Open Locks attempt?

RAG Answer:
If a Thief fails an Open Locks attempt, they must wait until they have gained another level of experience before trying again.

Model used: granite-3-2-8b-instruct
Context chunks used: 3


## 4.7 The Comparison: Baseline vs. RAG

**This is why we established a baseline in Section 2.**

Without "before," you can't prove "after." Without a baseline, every improvement is anecdotal. With one, it's evidence.

We'll ask the same questions with and without retrieval and compare results side by side.

> **Facilitator note:** This is the key teaching moment. Pause here. Let participants read both answers. Ask: *"Which answer would you trust in front of your legal team?"*


In [29]:
questions = [
    "What happens if a Thief fails an Open Locks attempt?",
    "Why can't Elves roll higher than a d6 for hit points?",
    "What is the maximum number of retainers a character can have?",
    "How does turning undead work for Clerics?",
]

headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}

print("=" * 80)
print("BASELINE vs RAG COMPARISON")
print("=" * 80)

for q in questions:
    print(f"\n{'─' * 80}")
    print(f"QUESTION: {q}")
    print(f"{'─' * 80}")
    
    # Baseline: no context
    baseline_data = {
        "model": "granite-3-2-8b-instruct",
        "messages": [{"role": "user", "content": q}],
        "max_tokens": 200,
        "temperature": 0
    }
    try:
        baseline_resp = requests.post(url_chat, headers=headers, json=baseline_data)
        baseline_answer = baseline_resp.json()["choices"][0]["message"]["content"]
    except Exception as e:
        baseline_answer = f"[Error: {e}]"
    
    # RAG: with retrieved context
    try:
        rag_result = rag_query(q, collection, embed_model, key, endpoint_base)
        rag_answer = rag_result["answer"]
    except Exception as e:
        rag_answer = f"[Error: {e}]"
    
    print(f"\n📭 BASELINE (no context):\n{baseline_answer}")
    print(f"\n📬 RAG (with retrieval):\n{rag_answer}")

print(f"\n{'=' * 80}")
print("Review: Which answers improved? Which didn't? Why?")
print(f"{'=' * 80}")

BASELINE vs RAG COMPARISON

────────────────────────────────────────────────────────────────────────────────
QUESTION: What happens if a Thief fails an Open Locks attempt?
────────────────────────────────────────────────────────────────────────────────

📭 BASELINE (no context):
If a Thief fails an Open Locks attempt, they do not make any progress towards opening the lock. The Thief must wait until their next attempt to try again. If the Thief fails by 4 or more, they may trigger a trap or alert nearby enemies.

📬 RAG (with retrieval):
If a Thief fails an Open Locks attempt, they must wait until they have gained another level of experience before trying again.

────────────────────────────────────────────────────────────────────────────────
QUESTION: Why can't Elves roll higher than a d6 for hit points?
────────────────────────────────────────────────────────────────────────────────

📭 BASELINE (no context):
In many fantasy role-playing games, such as Dungeons & Dragons, Elves are typic

## 4.8 Making Answers Defensible: Show the Evidence

A correct answer isn't enough. The customer needs to **explain why the system said what it said.**

This is one of the key advantages of RAG over fine-tuning: you can always point to the exact source chunks that informed the answer. This makes the system **auditable.**

> "Would you trust this in front of your legal / compliance / engineering team?"

If you can show the source, the answer to that question changes.


In [30]:
question = "What happens if a Thief fails an Open Locks attempt?"
result = rag_query(question, collection, embed_model, key, endpoint_base)

print(f"Question: {result['question']}\n")
print(f"Answer: {result['answer']}\n")
print("=" * 60)
print("EVIDENCE — Retrieved chunks that informed this answer:")
print("=" * 60)
for i, chunk in enumerate(result['retrieved_chunks']):
    print(f"\n--- Source Chunk {i+1} ---")
    print(chunk)

Question: What happens if a Thief fails an Open Locks attempt?

Answer: If a Thief fails an Open Locks attempt, they must wait until they have gained another level of experience before trying again.

EVIDENCE — Retrieved chunks that informed this answer:

--- Source Chunk 1 ---
ttempt fails, the Thief must wait until they have gained another level of experience before trying again.

Remove Traps is generally rolled twice: first to detect the trap, and second to disarm it.   The GM will make these rolls as the player won't know for sure if the character is successful or not until someone actually tests the trapped (or suspected) area.

Pick Pockets allows the Thief to lift the wallet, cut the purse, etc. of a victim without being noticed.   If the roll fails, the Thief didn't get what they wanted; but further, the intended victim (or an onlooker, at the GM's option) will notice the attempt if the die roll is more than two times the target number (or if the die roll is 00).

Move Silentl

## 4.9 Optional: How Chunking Strategy Changes Answers

If time permits, this exercise demonstrates that chunking strategy **directly impacts answer quality.** Same question, same model, same embedding model — different chunk sizes produce different retrieval results, and therefore different answers.

> **Facilitator note:** This is optional but powerful. If you're short on time, describe the tradeoff verbally and move on.


In [31]:
question = "What happens if a Thief fails an Open Locks attempt?"

for chunk_size in [200, 500, 1000]:
    print(f"\n{'=' * 60}")
    print(f"CHUNK SIZE: {chunk_size} characters (overlap: {int(chunk_size * 0.2)})")
    print(f"{'=' * 60}")
    
    # Re-chunk
    test_chunks = chunk_text(full_text, chunk_size=chunk_size, overlap=int(chunk_size * 0.2))
    
    # Embed
    test_embeddings = embed_model.encode(test_chunks, show_progress_bar=False, batch_size=64)
    
    # Create a fresh collection
    coll_name = f"rpg_experiment_{chunk_size}"
    try:
        client.delete_collection(coll_name)
    except:
        pass
    
    test_collection = client.create_collection(
        name=coll_name,
        metadata={"hnsw:space": "cosine"}
    )
    
    test_collection.add(
        documents=test_chunks,
        embeddings=test_embeddings.tolist(),
        ids=[f"chunk_{i}" for i in range(len(test_chunks))]
    )
    
    # Query
    result = rag_query(question, test_collection, embed_model, key, endpoint_base)
    print(f"Total chunks in store: {len(test_chunks)}")
    print(f"Answer: {result['answer']}")
    
    # Cleanup
    client.delete_collection(coll_name)


CHUNK SIZE: 200 characters (overlap: 40)


InternalError: ValueError: Batch size of 5681 is greater than max batch size of 5461

## 4.10 Decision Point: Do We Need More Than RAG?

We've now seen what RAG can do. Same model, dramatically better answers — because we gave it the right context.

**RAG works when:**
- Facts are explicit in the documents
- Language is stable and consistent
- Reasoning required is relatively shallow (look up and report)

**RAG struggles when:**
- Terminology is domain-specific and the model doesn't understand it
- Rules are implicit — they require inference, not just lookup
- Meaning depends on context that spans multiple documents or editions
- The model has the right context and *still* gets it wrong

> **Key question to ask:** *"Is the model failing because it doesn't have the information, or because it doesn't understand the information?"*

If the answer is "doesn't have" → improve retrieval.  
If the answer is "doesn't understand" → that's when model customization enters the conversation.

But we don't guess. We **evaluate.**

---

*"We now understand why the model behaves the way it does. Next, we decide how far to escalate."*

**Next: Section 5 — Evaluation, Escalation, and Knowing When to Change the Model**


---

## Appendix: Using MaaS Embedding Endpoint (Alternative to Local Model)

If your MaaS instance has `nomic-embed-text-v1-5` available, you can use the API for embeddings instead of the local model. Replace the embedding setup cells with the code below.


In [ ]:
# ALTERNATIVE: Use MaaS embedding endpoint instead of local model
# Uncomment and use this if nomic-embed-text-v1-5 is available

# class MaaSEmbedder:
#     """Drop-in replacement for SentenceTransformer that calls MaaS API."""
#     def __init__(self, api_key, base_url, model="nomic-embed-text-v1-5"):
#         self.api_key = api_key
#         self.base_url = base_url
#         self.model = model
# 
#     def encode(self, texts, **kwargs):
#         import numpy as np
#         url = f"{self.base_url}/embeddings"
#         headers = {
#             "Authorization": f"Bearer {self.api_key}",
#             "Content-Type": "application/json"
#         }
#         # Batch in groups of 50
#         all_embeddings = []
#         for i in range(0, len(texts), 50):
#             batch = texts[i:i+50]
#             data = {"model": self.model, "input": batch}
#             response = requests.post(url, headers=headers, json=data)
#             response.raise_for_status()
#             result = response.json()
#             all_embeddings.extend([item["embedding"] for item in result["data"]])
#         return np.array(all_embeddings)
#
# # Use this instead of SentenceTransformer:
# embed_model = MaaSEmbedder(api_key=key, base_url=endpoint_base)

---

## Facilitator Reference: Pre-Built Outputs

If live execution stalls, use these expected outputs to keep the session moving.

**Chunk count** (500 char, 100 overlap): ~2,200 chunks (depends on document length)

**Retrieval for "Open Locks" question:** Should return chunks containing the Thief class abilities section, specifically:

> *"Open Locks allows the Thief to unlock a lock without a proper key. It may only be tried once per lock. If the attempt fails, the Thief must wait until they have gained another level of experience before trying again."*

**RAG answer** should closely match the source text above.

**Baseline answer** will reference D&D or generic RPG rules — notably wrong for this game system.

**Side-by-side comparison** should show clear improvement for questions with explicit answers in the rulebook, and may show less improvement for questions requiring implicit reasoning (which sets up Section 5's evaluation discussion).

> *"We'll use the prepared results here so we can focus on what matters."*  
> — This is expected behavior, not a failure mode.
